# Unified CONDOR/FlexDC Training Notebook

This notebook trains the unchanged CONDOR `DataCenterModel` on FlexDC-generated data using one of four target definitions:

1. `condor / normal`
2. `condor / raw`
3. `flexdc / normal`
4. `flexdc / raw`

The same combined FlexDC `grid_search_results.csv` and `grid_search_diagnostics.csv` are used for every variant. The targets are constructed inside `am_unified_training_utilities.py`, so do not create separate cleaned/converted datasets for different model families.

**Cell labels:**
- `[COLAB ONLY]` cells should be run in Google Colab.
- `[LOCAL PC ONLY]` cells are for local Jupyter/VS Code runs, not Colab.
- `[ALL]` cells should be run in either environment after paths are set.


## 0. Choose environment

**Run this cell everywhere.**

Set `RUN_ENV = 'colab'` when using Google Colab. Set `RUN_ENV = 'local_pc'` only if you are running this notebook from your Windows machine with the repo already on disk.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

RUN_ENV = 'colab'  # 'colab' or 'local_pc'

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or Path('/content').exists()
print('Detected Colab-like environment:', IN_COLAB)
print('RUN_ENV:', RUN_ENV)


Detected Colab-like environment: True
RUN_ENV: colab


## 1. [COLAB ONLY] Install minimal dependencies

Run this cell in **Google Colab only**.

Do **not** run this on your local Conda environment unless you intentionally want to install packages into the current notebook kernel. Colab usually already has PyTorch installed, so we only install the smaller packages needed for W&B, data handling, and plots.


In [ ]:
if RUN_ENV == 'colab':
    %pip install -q wandb pandas numpy scikit-learn tqdm matplotlib
else:
    print('SKIP: local_pc mode. Do not pip-install from this notebook.')

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


Torch: 2.11.0+cu128
CUDA available: True


## 2. [COLAB ONLY] Clone or update the GitHub repo

Run this cell in **Google Colab only**.

Opening a notebook from GitHub does **not** make the full repository available in the Colab runtime. You still need to clone the repo here.

Before running, set `COMDER_REPO_URL` to your real GitHub URL. If the repo is private, use the GitHub URL format/token approach you normally use for private Colab clones.


In [ ]:
# ===== EDIT THESE =====
COMDER_REPO_URL = 'https://github.com/NetherMoon/CONDOR-FLEXDC'
COMDER_BRANCH = 'main'
# ======================

if RUN_ENV == 'colab':
    if COMDER_REPO_URL.startswith('PUT_'):
        raise ValueError('Set COMDER_REPO_URL to your actual GitHub repo URL before running this cell.')

    WORKSPACE = Path('/content/workspace')
    COMDER_ROOT = WORKSPACE / 'comder-main'
    WORKSPACE.mkdir(parents=True, exist_ok=True)

    if COMDER_ROOT.exists():
        print('Repo already exists. Pulling latest changes...')
        subprocess.run(['git', '-C', str(COMDER_ROOT), 'fetch'], check=True)
        subprocess.run(['git', '-C', str(COMDER_ROOT), 'checkout', COMDER_BRANCH], check=True)
        subprocess.run(['git', '-C', str(COMDER_ROOT), 'pull'], check=True)
    else:
        print('Cloning repo...')
        subprocess.run(['git', 'clone', '--branch', COMDER_BRANCH, COMDER_REPO_URL, str(COMDER_ROOT)], check=True)

    print('COMDER_ROOT:', COMDER_ROOT)
else:
    print('SKIP: local_pc mode. Use the local path setup cell instead.')


Cloning repo...
COMDER_ROOT: /content/workspace/comder-main


## 3. [COLAB OPTIONAL] Copy dataset from Google Drive

Run this cell **only if the new combined dataset is not committed to GitHub**.

Set `USE_GOOGLE_DRIVE_DATA = True` and update the two Drive paths. The cell copies the two combined CSVs into the expected repo location:

`am_flexdc/data/pilots/traditionaliso_newqos_pilot/`

If the dataset is already in the repo, leave this cell as-is and it will skip.


In [ ]:
USE_GOOGLE_DRIVE_DATA = False

# Update these only if USE_GOOGLE_DRIVE_DATA = True
DRIVE_RESULTS_CSV = '/content/drive/MyDrive/path/to/traditional_iso16_newqos_AQA_combined_grid_search_results.csv'
DRIVE_DIAGNOSTICS_CSV = '/content/drive/MyDrive/path/to/traditional_iso16_newqos_AQA_combined_grid_search_diagnostics.csv'

if RUN_ENV == 'colab' and USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive
    drive.mount('/content/drive')

    pilot_dir = COMDER_ROOT / 'am_flexdc' / 'data' / 'pilots' / 'traditionaliso_newqos_pilot_flexdc_configured_objective'
    pilot_dir.mkdir(parents=True, exist_ok=True)

    subprocess.run(['cp', DRIVE_RESULTS_CSV, str(pilot_dir / 'traditional_iso16_newqos_AQA_combined_grid_search_results.csv')], check=True)
    subprocess.run(['cp', DRIVE_DIAGNOSTICS_CSV, str(pilot_dir / 'traditional_iso16_newqos_AQA_combined_grid_search_diagnostics.csv')], check=True)

    print('Copied dataset into:', pilot_dir)
else:
    print('SKIP: Drive copy not requested.')


SKIP: Drive copy not requested.


## 4. [LOCAL PC ONLY] Set local Windows paths

Run this cell **only if you are using this notebook locally on your PC**.

Do **not** run this in Colab. For Colab, the clone cell above sets `COMDER_ROOT`.


In [ ]:
if RUN_ENV == 'local_pc':
    COMDER_ROOT = Path(r'C:\Users\Achuthan Menon\Desktop\Research Work\comder-main')
    if not COMDER_ROOT.exists():
        raise FileNotFoundError(f'COMDER_ROOT does not exist: {COMDER_ROOT}')
    print('Local COMDER_ROOT:', COMDER_ROOT)
else:
    print('SKIP: Colab mode. COMDER_ROOT should already come from the clone cell.')


## 5. [ALL] Resolve paths and check required files

Run this cell after either the Colab clone path is set or the local PC path is set.

Edit `PILOT_NAME` and `PILOT_PREFIX` if your combined dataset folder or filenames change.


In [ ]:
# Dataset naming from the recomputed configured-objective new pilot
PILOT_NAME = 'traditionaliso_newqos_pilot_flexdc_configured_objective'
PILOT_PREFIX = 'traditional_iso16_newqos_AQA'

AM_FLEXDC_ROOT = COMDER_ROOT / 'am_flexdc'
TRAIN_DIR = AM_FLEXDC_ROOT / 'train'
MODELS_DIR = AM_FLEXDC_ROOT / 'models'
RESULTS_DIR = AM_FLEXDC_ROOT / 'results' / 'training_runs'
PILOT_DIR = AM_FLEXDC_ROOT / 'data' / 'pilots' / PILOT_NAME

RESULTS_CSV = PILOT_DIR / f'{PILOT_PREFIX}_combined_grid_search_results.csv'
DIAGNOSTICS_CSV = PILOT_DIR / f'{PILOT_PREFIX}_combined_grid_search_diagnostics.csv'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    TRAIN_DIR / 'data_center_model.py',
    TRAIN_DIR / 'am_unified_training_utilities.py',
    RESULTS_CSV,
    DIAGNOSTICS_CSV,
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    print('Missing required files:')
    for p in missing:
        print(' -', p)
    raise FileNotFoundError('Fix missing paths before training.')

print('Paths OK')
print('TRAIN_DIR:', TRAIN_DIR)
print('RESULTS_CSV:', RESULTS_CSV)
print('DIAGNOSTICS_CSV:', DIAGNOSTICS_CSV)
print('MODELS_DIR:', MODELS_DIR)
print('RESULTS_DIR:', RESULTS_DIR)


Paths OK
TRAIN_DIR: /content/workspace/comder-main/am_flexdc/train
RESULTS_CSV: /content/workspace/comder-main/am_flexdc/data/pilots/traditionaliso_newqos_pilot/traditional_iso16_newqos_AQA_combined_grid_search_results.csv
DIAGNOSTICS_CSV: /content/workspace/comder-main/am_flexdc/data/pilots/traditionaliso_newqos_pilot/traditional_iso16_newqos_AQA_combined_grid_search_diagnostics.csv
MODELS_DIR: /content/workspace/comder-main/am_flexdc/models
RESULTS_DIR: /content/workspace/comder-main/am_flexdc/results/training_runs


## 5b. [ALL] Validate configured objective columns

This checks that the dataset columns being used for FlexDC-normal labels have been recomputed with the configured objective parameters from the reference simulated-annealing cost configuration. It does not change the CSVs; it only validates them before training/inference.


In [ ]:
import ast
import json
import numpy as np
import pandas as pd

FLEXDC_OBJECTIVE_CONSTANTS = {
    "ctrack_psi": 1.0,
    "ctrack_mu": 10.0,
    "ctrack_gamma": 0.3,
    "qos_beta": 20.0,
    "qos_rho": 2.0,
    "qos_threshold": 0.1,
}


def _stable_softplus(x):
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _parse_probability_vector(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=float)
    if pd.isna(value):
        raise ValueError("QoS_Delay_Probabilities contains NaN")
    text = str(value).strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        parsed = ast.literal_eval(text)
    arr = np.asarray(parsed, dtype=float)
    if arr.ndim != 1:
        raise ValueError(f"Expected 1-D QoS probability vector, got {arr.shape}")
    return arr


def validate_configured_objective_columns(results_csv, diagnostics_csv, tol=1e-8):
    result_needed = {
        "Simulator_RSR_Total_Cost",
        "QoS_Delay_Probabilities",
        "Ctrack_Epsilon_90th",
        "Ctrack_Gamma",
        "Ctrack_Psi",
        "Ctrack_Mu",
        "Ctrack_Weighted_Cost",
        "Diagnostic_FlexDC_SoftPlus_QoS_Cost",
        "Diagnostic_FullPaperObjective_Cost",
    }
    diag_needed = {
        "Ctrack_Gamma",
        "Ctrack_Psi",
        "Ctrack_Mu",
        "Ctrack_Weighted_Cost",
        "Diagnostic_FlexDC_SoftPlus_QoS_Cost",
        "Diagnostic_FullPaperObjective_Cost",
    }

    res = pd.read_csv(results_csv, usecols=lambda c: c in result_needed, low_memory=False)
    diag = pd.read_csv(diagnostics_csv, usecols=lambda c: c in diag_needed, low_memory=False)

    missing_res = sorted(result_needed - set(res.columns))
    missing_diag = sorted(diag_needed - set(diag.columns))
    if missing_res or missing_diag:
        raise KeyError(f"Missing required columns. results missing={missing_res}, diagnostics missing={missing_diag}")
    if len(res) != len(diag):
        raise ValueError(f"Row-count mismatch: results={len(res)}, diagnostics={len(diag)}")

    c = FLEXDC_OBJECTIVE_CONSTANTS
    for col, expected in [
        ("Ctrack_Psi", c["ctrack_psi"]),
        ("Ctrack_Mu", c["ctrack_mu"]),
        ("Ctrack_Gamma", c["ctrack_gamma"]),
    ]:
        max_diff_res = float((res[col].astype(float) - expected).abs().max())
        max_diff_diag = float((diag[col].astype(float) - expected).abs().max())
        if max(max_diff_res, max_diff_diag) > tol:
            raise AssertionError(f"{col} does not match expected {expected}: results diff={max_diff_res}, diagnostics diff={max_diff_diag}")

    eps = res["Ctrack_Epsilon_90th"].astype(float).to_numpy()
    expected_ctrack = c["ctrack_psi"] * _stable_softplus(c["ctrack_mu"] * (eps - c["ctrack_gamma"]))

    probs = [_parse_probability_vector(v) for v in res["QoS_Delay_Probabilities"]]
    expected_qos = np.asarray([
        c["qos_beta"] * float(np.sum(_stable_softplus(c["qos_rho"] * (p - c["qos_threshold"]))))
        for p in probs
    ], dtype=float)

    expected_full = res["Simulator_RSR_Total_Cost"].astype(float).to_numpy() + expected_ctrack + expected_qos

    checks = {
        "results Ctrack reconstruction": float(np.max(np.abs(res["Ctrack_Weighted_Cost"].astype(float).to_numpy() - expected_ctrack))),
        "results QoS reconstruction": float(np.max(np.abs(res["Diagnostic_FlexDC_SoftPlus_QoS_Cost"].astype(float).to_numpy() - expected_qos))),
        "results full reconstruction": float(np.max(np.abs(res["Diagnostic_FullPaperObjective_Cost"].astype(float).to_numpy() - expected_full))),
        "results-vs-diagnostics Ctrack": float(np.max(np.abs(res["Ctrack_Weighted_Cost"].astype(float).to_numpy() - diag["Ctrack_Weighted_Cost"].astype(float).to_numpy()))),
        "results-vs-diagnostics QoS": float(np.max(np.abs(res["Diagnostic_FlexDC_SoftPlus_QoS_Cost"].astype(float).to_numpy() - diag["Diagnostic_FlexDC_SoftPlus_QoS_Cost"].astype(float).to_numpy()))),
        "results-vs-diagnostics full": float(np.max(np.abs(res["Diagnostic_FullPaperObjective_Cost"].astype(float).to_numpy() - diag["Diagnostic_FullPaperObjective_Cost"].astype(float).to_numpy()))),
    }

    bad = {name: diff for name, diff in checks.items() if diff > tol}
    if bad:
        raise AssertionError(f"Configured objective validation failed: {bad}")

    summary = pd.DataFrame([
        {"Check": name, "Max absolute difference": diff} for name, diff in checks.items()
    ])
    print("Configured FlexDC objective validation passed.")
    print("Constants:", FLEXDC_OBJECTIVE_CONSTANTS)
    print(f"Rows checked: {len(res):,}")
    display(summary)
    return summary

configured_objective_validation = validate_configured_objective_columns(RESULTS_CSV, DIAGNOSTICS_CSV)


## 6. [ALL] Import training code

Run this cell in both Colab and local environments. It adds `am_flexdc/train` to `sys.path` so imports work correctly.


In [ ]:
import os
import sys

os.chdir(TRAIN_DIR)
if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

from data_center_model import DataCenterModel
from am_unified_training_utilities import DCDataset, train_model, evaluate_model

print('Current directory:', Path.cwd())
print('Imports OK')


Current directory: /content/workspace/comder-main/am_flexdc/train
Imports OK


## 7. [ALL] W&B login

Run this cell if you want W&B logging.

Set `USE_WANDB = False` if you want to train without W&B.


In [ ]:
USE_WANDB = True

if USE_WANDB:
    import wandb
    wandb.login()
else:
    wandb = None
    print('W&B disabled for this notebook run.')


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amenon06 (amenon06-boston-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 8. [ALL] Choose one model variant

Run this cell before training. Change only `TARGET_FAMILY` and `TARGET_MODE` to switch among the four requested models.

Valid combinations:
- `condor / normal`
- `condor / raw`
- `flexdc / normal`
- `flexdc / raw`


In [ ]:
TARGET_FAMILY = 'flexdc'   # 'condor' or 'flexdc'
TARGET_MODE = 'normal'     # 'normal' or 'raw'

USE_NORM_PR = True
USE_NORM_COST = None       # None = CONDOR uses released scaling; FlexDC stays in logged units
USE_NORM_WLMIX = True
RAW_QOS_AGGREGATION = 'mean'  # 'mean' or 'sum'; confirm with Fatih/Kerim if needed

EPOCHS = 150
LR = 1e-4
BATCH_SIZE = 512
CROSS_VALIDATE = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

WANDB_ENTITY = 'amenon06-boston-university'
WANDB_PROJECT = 'flexdc-unified-training'
DATASET_TAG = 'newqos_configured_objective_v1'
RUN_NAME = f'{TARGET_FAMILY}_{TARGET_MODE}_{DATASET_TAG}'

MODEL_DIR = MODELS_DIR / f'{TARGET_FAMILY}_{TARGET_MODE}'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_DIR / f'am_{TARGET_FAMILY}_{TARGET_MODE}_{DATASET_TAG}_state_dict.pt'
METRICS_FILE = RESULTS_DIR / f'am_{TARGET_FAMILY}_{TARGET_MODE}_{DATASET_TAG}_metrics.csv'

print('Run:', RUN_NAME)
print('Device:', DEVICE)
print('Model will save to:', MODEL_FILE)
print('Metrics will save to:', METRICS_FILE)


Run: flexdc_normal_newqos_pilot_v1
Device: cuda
Model will save to: /content/workspace/comder-main/am_flexdc/models/flexdc_normal/am_flexdc_normal_newqos_pilot_v1_state_dict.pt
Metrics will save to: /content/workspace/comder-main/am_flexdc/results/training_runs/am_flexdc_normal_newqos_pilot_v1_metrics.csv


## 9. [ALL] Dataset sanity check before training

Run this before training. It verifies that the required columns exist and prints target distributions for the chosen variant.


In [ ]:
dataset = DCDataset(
    results_csv=RESULTS_CSV,
    diagnostics_csv=DIAGNOSTICS_CSV,
    target_family=TARGET_FAMILY,
    target_mode=TARGET_MODE,
    use_norm_pr=USE_NORM_PR,
    use_norm_cost=USE_NORM_COST,
    use_norm_wlmix=USE_NORM_WLMIX,
    raw_qos_aggregation=RAW_QOS_AGGREGATION,
)
dataset.get_statistics()


*** Datacenter Dataset Statistics ***
Rows: 161600
Target family/mode: flexdc/normal
Target names: ['flexdc_M_RSR', 'flexdc_Ctrack_weighted', 'flexdc_CQoS_weighted']
flexdc_M_RSR: mean=22.3247, std=23.5213, min=0.0173668, max=103.937
flexdc_Ctrack_weighted: mean=12.6551, std=25.1593, min=4.43484, max=276.376
flexdc_CQoS_weighted: mean=2.01702, std=0.948317, min=1.19628, max=3.77718
Workload norm weights: [2.00375e+02 7.17000e+02 5.00125e+02 2.01900e+03 4.06250e+00 1.00000e+00
 1.00000e+00]


## 10. [ALL] Train one model

Run this cell to train one selected model variant.

Do **not** run the all-four cell below at the same time unless you intentionally want to train all models.


In [ ]:
config = {
    'target_family': TARGET_FAMILY,
    'target_mode': TARGET_MODE,
    'use_norm_pr': USE_NORM_PR,
    'use_norm_cost': USE_NORM_COST,
    'use_norm_wlmix': USE_NORM_WLMIX,
    'raw_qos_aggregation': RAW_QOS_AGGREGATION,
    'epochs': EPOCHS,
    'lr': LR,
    'batch_size': BATCH_SIZE,
    'cross_validate': CROSS_VALIDATE,
    'device': DEVICE,
    'dataset_tag': DATASET_TAG,
    'results_csv': str(RESULTS_CSV),
    'diagnostics_csv': str(DIAGNOSTICS_CSV),
    'flexdc_objective_constants': FLEXDC_OBJECTIVE_CONSTANTS,
}

run = None
if USE_WANDB:
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=RUN_NAME,
        config=config,
    )

model = DataCenterModel()
model, train_losses, heldout_losses, target_names = train_model(
    model,
    epochs=EPOCHS,
    lr=LR,
    batch_size=BATCH_SIZE,
    verbose=True,
    cross_validate=CROSS_VALIDATE,
    results_csv=RESULTS_CSV,
    diagnostics_csv=DIAGNOSTICS_CSV,
    target_family=TARGET_FAMILY,
    target_mode=TARGET_MODE,
    use_norm_pr=USE_NORM_PR,
    use_norm_cost=USE_NORM_COST,
    use_norm_wlmix=USE_NORM_WLMIX,
    raw_qos_aggregation=RAW_QOS_AGGREGATION,
    device_name=DEVICE,
    wandb_run=run,
)

torch.save(model.state_dict(), MODEL_FILE)
if run is not None:
    run.save(str(MODEL_FILE))
print('Saved model:', MODEL_FILE)


Epoch 0 Train Loss: 337.38487940999715 Test Loss: 331.563961952611
Epoch 1 Train Loss: 332.1251822769372 Test Loss: 330.7363112600226
Epoch 2 Train Loss: 329.7480328590082 Test Loss: 325.9571737189042
Epoch 3 Train Loss: 312.61944048437056 Test Loss: 281.08658334832444
Epoch 4 Train Loss: 250.30609952486478 Test Loss: 213.60423423365543
Epoch 5 Train Loss: 206.72680139325863 Test Loss: 186.47494257876747
Epoch 6 Train Loss: 182.0313515986792 Test Loss: 175.32032984683389
Epoch 7 Train Loss: 169.47770432001866 Test Loss: 166.92866644608347
Epoch 8 Train Loss: 164.00837565978728 Test Loss: 155.4305075796027
Epoch 9 Train Loss: 157.4411557055167 Test Loss: 152.63342903538754
Epoch 10 Train Loss: 150.1732242463401 Test Loss: 147.19005383943258
Epoch 11 Train Loss: 144.6644034752479 Test Loss: 140.8062818828382
Epoch 12 Train Loss: 135.76347361430862 Test Loss: 151.2624690808748
Epoch 13 Train Loss: 127.48858421636383 Test Loss: 120.23477381656045
Epoch 14 Train Loss: 117.24123203053193 Tes

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 149 Train Loss: 11.892074291522686 Test Loss: 9.426252831910785
Training Time: 650.9944920539856
Metric Tons of Co2: 4.177248075063666e-05
Saved model: /content/workspace/comder-main/am_flexdc/models/flexdc_normal/am_flexdc_normal_newqos_pilot_v1_state_dict.pt


## 11. [ALL] Evaluate and save metrics

Run this after training one model. It saves a one-row metrics CSV and logs final metrics to W&B if enabled.


In [ ]:
import pandas as pd

y_true_train, y_pred_train, y_true_test, y_pred_test, metrics, target_names = evaluate_model(
    model,
    cross_validate=CROSS_VALIDATE,
    batch_size=BATCH_SIZE,
    results_csv=RESULTS_CSV,
    diagnostics_csv=DIAGNOSTICS_CSV,
    target_family=TARGET_FAMILY,
    target_mode=TARGET_MODE,
    use_norm_pr=USE_NORM_PR,
    use_norm_cost=USE_NORM_COST,
    use_norm_wlmix=USE_NORM_WLMIX,
    raw_qos_aggregation=RAW_QOS_AGGREGATION,
    device_name=DEVICE,
    print_metrics=True,
)

metrics_row = {
    'run_name': RUN_NAME,
    'target_family': TARGET_FAMILY,
    'target_mode': TARGET_MODE,
    'model_file': str(MODEL_FILE),
    'target_names': ','.join(target_names),
    **metrics,
}
metrics_df = pd.DataFrame([metrics_row])
metrics_df.to_csv(METRICS_FILE, index=False)
display(metrics_df.T)

if run is not None:
    run.log(metrics)
    run.summary.update(metrics)
    run.finish()

print('Saved metrics:', METRICS_FILE)


Target names: ['flexdc_M_RSR', 'flexdc_Ctrack_weighted', 'flexdc_CQoS_weighted']
MSE  || Train: 9.724809646606445 | Test: 9.431174278259277
MAPE || Train: 0.312313437461853 | Test: 0.3174149692058563
Target-sum MSE  || Train: 33.17725372314453 | Test: 32.2613639831543
Target-sum MAPE || Train: 0.12003128230571747 | Test: 0.11993680894374847
flexdc_M_RSR | heldout MAE=1.75958 RMSE=2.608 R2=0.987763
flexdc_Ctrack_weighted | heldout MAE=1.92309 RMSE=4.62845 R2=0.9659
flexdc_CQoS_weighted | heldout MAE=0.194028 RMSE=0.263896 R2=0.922713


,0
run_name,flexdc_normal_newqos_pilot_v1
target_family,flexdc
target_mode,normal
model_file,/content/workspace/comder-main/am_flexdc/model...
target_names,"flexdc_M_RSR,flexdc_Ctrack_weighted,flexdc_CQo..."
train/flexdc_M_RSR_mae,1.757756
train/flexdc_M_RSR_rmse,2.603045
train/flexdc_M_RSR_r2,0.987728
train/flexdc_Ctrack_weighted_mae,1.945752
train/flexdc_Ctrack_weighted_rmse,4.725575


epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇██
heldout/component_mape,▁
heldout/component_mse,▁
heldout/flexdc_CQoS_weighted_mae,▁
heldout/flexdc_CQoS_weighted_r2,▁
heldout/flexdc_CQoS_weighted_rmse,▁
heldout/flexdc_Ctrack_weighted_mae,▁
heldout/flexdc_Ctrack_weighted_r2,▁
heldout/flexdc_Ctrack_weighted_rmse,▁
heldout/flexdc_M_RSR_mae,▁
+25,...


Saved metrics: /content/workspace/comder-main/am_flexdc/results/training_runs/am_flexdc_normal_newqos_pilot_v1_metrics.csv


In [ ]:
import shutil
from pathlib import Path

# Define the source directory and the output filename
source_dir = '/content/workspace/comder-main/am_flexdc/models'
output_filename = 'models_archive'

if Path(source_dir).exists():
    # Create a zip archive of the directory
    shutil.make_archive(output_filename, 'zip', source_dir)
    print(f'Successfully zipped {source_dir} into {output_filename}.zip')
else:
    print(f'Error: Source directory {source_dir} does not exist.')


Successfully zipped /content/workspace/comder-main/am_flexdc/models into models_archive.zip


## 12. [ALL / OPTIONAL] Train all four variants

Only run this if you want to train all four requested models in one session.

Leave `RUN_ALL_FOUR = False` unless you are ready. This cell creates four W&B runs and four checkpoints.


In [ ]:
RUN_ALL_FOUR = False

if RUN_ALL_FOUR:
    all_rows = []
    variants = [
        ('condor', 'normal'),
        ('condor', 'raw'),
        ('flexdc', 'normal'),
        ('flexdc', 'raw'),
    ]

    for target_family, target_mode in variants:
        run_name = f'{target_family}_{target_mode}_{DATASET_TAG}'
        model_dir = MODELS_DIR / f'{target_family}_{target_mode}'
        model_dir.mkdir(parents=True, exist_ok=True)
        model_file = model_dir / f'am_{target_family}_{target_mode}_{DATASET_TAG}_state_dict.pt'
        metrics_file = RESULTS_DIR / f'am_{target_family}_{target_mode}_{DATASET_TAG}_metrics.csv'

        config = {
            'target_family': target_family,
            'target_mode': target_mode,
            'use_norm_pr': USE_NORM_PR,
            'use_norm_cost': USE_NORM_COST,
            'use_norm_wlmix': USE_NORM_WLMIX,
            'raw_qos_aggregation': RAW_QOS_AGGREGATION,
            'epochs': EPOCHS,
            'lr': LR,
            'batch_size': BATCH_SIZE,
            'cross_validate': CROSS_VALIDATE,
            'device': DEVICE,
            'dataset_tag': DATASET_TAG,
            'results_csv': str(RESULTS_CSV),
            'diagnostics_csv': str(DIAGNOSTICS_CSV),
            'flexdc_objective_constants': FLEXDC_OBJECTIVE_CONSTANTS,
        }

        this_run = None
        if USE_WANDB:
            this_run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=run_name, config=config)

        this_model = DataCenterModel()
        this_model, train_losses, heldout_losses, target_names = train_model(
            this_model,
            epochs=EPOCHS,
            lr=LR,
            batch_size=BATCH_SIZE,
            verbose=True,
            cross_validate=CROSS_VALIDATE,
            results_csv=RESULTS_CSV,
            diagnostics_csv=DIAGNOSTICS_CSV,
            target_family=target_family,
            target_mode=target_mode,
            use_norm_pr=USE_NORM_PR,
            use_norm_cost=USE_NORM_COST,
            use_norm_wlmix=USE_NORM_WLMIX,
            raw_qos_aggregation=RAW_QOS_AGGREGATION,
            device_name=DEVICE,
            wandb_run=this_run,
        )

        torch.save(this_model.state_dict(), model_file)
        if this_run is not None:
            this_run.save(str(model_file))

        y_true_train, y_pred_train, y_true_test, y_pred_test, metrics, target_names = evaluate_model(
            this_model,
            cross_validate=CROSS_VALIDATE,
            batch_size=BATCH_SIZE,
            results_csv=RESULTS_CSV,
            diagnostics_csv=DIAGNOSTICS_CSV,
            target_family=target_family,
            target_mode=target_mode,
            use_norm_pr=USE_NORM_PR,
            use_norm_cost=USE_NORM_COST,
            use_norm_wlmix=USE_NORM_WLMIX,
            raw_qos_aggregation=RAW_QOS_AGGREGATION,
            device_name=DEVICE,
            print_metrics=True,
        )

        row = {
            'run_name': run_name,
            'target_family': target_family,
            'target_mode': target_mode,
            'model_file': str(model_file),
            'target_names': ','.join(target_names),
            **metrics,
        }
        pd.DataFrame([row]).to_csv(metrics_file, index=False)
        all_rows.append(row)

        if this_run is not None:
            this_run.log(metrics)
            this_run.summary.update(metrics)
            this_run.finish()

    all_metrics = pd.DataFrame(all_rows)
    all_metrics_file = RESULTS_DIR / f'all_four_{DATASET_TAG}_metrics.csv'
    all_metrics.to_csv(all_metrics_file, index=False)
    display(all_metrics)
    print('Saved all metrics:', all_metrics_file)
else:
    print('RUN_ALL_FOUR is False; skipping all-four training.')


## 13. [COLAB OPTIONAL] Save/download outputs

Run this in **Colab only** after training if you want to download the trained checkpoint and metrics.

If you trained all four variants, it is usually easier to zip the whole `models/` and `results/training_runs/` folders.


In [ ]:
if RUN_ENV == 'colab':
    from google.colab import files

    # Download the current single-model checkpoint and metrics.
    if MODEL_FILE.exists():
        files.download(str(MODEL_FILE))
    if METRICS_FILE.exists():
        files.download(str(METRICS_FILE))
else:
    print('SKIP: not running in Colab.')


## 14. [COLAB OPTIONAL] Copy trained outputs to Google Drive

Run this in **Colab only** if you want to persist models to Drive instead of downloading manually.

Set `COPY_OUTPUTS_TO_DRIVE = True` and edit `DRIVE_OUTPUT_DIR`.


In [ ]:
COPY_OUTPUTS_TO_DRIVE = False
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/flexdc_unified_training_outputs'

if RUN_ENV == 'colab' and COPY_OUTPUTS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    subprocess.run(['cp', '-r', str(MODELS_DIR), DRIVE_OUTPUT_DIR], check=True)
    subprocess.run(['cp', '-r', str(RESULTS_DIR), DRIVE_OUTPUT_DIR], check=True)
    print('Copied outputs to:', DRIVE_OUTPUT_DIR)
else:
    print('SKIP: Drive output copy not requested.')
